# Injury Intelligence - RAG Pipeline

Author: Angela Lekivetz

This notebook builds the OSHA narratives into a FAISS (Facebook AI Similarity Search) to store and organize the embeddings. A retrieval function is then used to retrieve the most similar narratives from the FAISS index, and this information is then passed to Claude to generate a response to plain English text. 

No model training is required as we are using the `all-MiniLM-L6-v2` model.

In [1]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import anthropic
import os

/Users/lekivetza/.pyenv/versions/3.11.9/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load the Dataset

The cleaned and lemmatized dataset from `1_eda.ipynb` is loaded. 

Our RAG pipeline will be based on the `Abstract Text` column, as we want to give Claude the full, readable original text of the accident.


In [6]:
df = pd.read_csv('../data/osha_filtered.csv')
print(df.shape)
df.head()

(4427, 8)


,Abstract Text,Event type,Event Keywords,Nature of Injury,Part of Body,text_len,clean_text,lemma_text
0,"At 9:00 a.m. on August 10, 2017, an employee w...",Caught in or between,"FINGER,MECHANICAL POWER PRESS,AMPUTATION,GUARD","Amputation, Crushing",Fingers,73,an employee was operating ton bliss coin knuck...,employee operate ton bliss coin knuckle mechan...
1,"At 9:45 a.m. on July 17, 2017, an employee was...",Caught in or between,"CAUGHT IN,DRIVE SHAFT,RESIDENTIAL CONSTRUCTION...",Dislocation,Fingers,107,an employee was using battery operated drill t...,employee battery operate drill drill hole wood...
2,"At 7:30 a.m. on June 30, 2017, an employee was...",Other,"AMPUTATED,EXPLOSION,FIREWORKS",Fire Burn,Hand,29,an employee was inserting match fuze into fire...,employee insert match fuze firework charge fir...
3,"At 2:00 p.m. on June 30, 2017, an employee was...",Fall (from elevation),"RIB,ROOF,HEAD,FALL PROTECTION,FALL,COLLARBONE,...",Serious Fall/Strike,Head,121,an employee was installing self adhering membr...,employee instal self adhere membrane roof slop...
4,"At 12:20 p.m. on June 23, 2017, an employee wa...",Struck-by,"STRUCK BY,TRUCK,BRAIN,NECK,FRACTURE,UNSTABLE LOAD","Bruising, Contusion",Neck,53,an employee was delivering load of plywood to ...,employee deliver load plywood project site emp...


## 2. Build Embeddings

In this section, we will build embeddings for the dataset using the `all-MiniLM-L6-v2` model. This converts the narratives into vector representations, enabling us to determine the similarity between them.

We then store the embeddings using FAISS, a library for efficient similarity search and clustering.

In [7]:
EMBED_MODEL = 'all-MiniLM-L6-v2'
embedder = SentenceTransformer(EMBED_MODEL)

print('Encoding texts...')
embeddings = embedder.encode(df['Abstract Text'].tolist(), show_progress_bar=True, batch_size=64)
embeddings = embeddings.astype('float32')
print(f'Embeddings shape: {embeddings.shape}')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9209.60it/s]


Encoding texts...


Batches: 100%|██████████| 70/70 [00:08<00:00,  8.42it/s]


Embeddings shape: (4427, 384)


In [9]:
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)
print(f'Index contains {index.ntotal} vectors')

faiss.write_index(index, '../data/injury_index.faiss')
df.to_csv('../data/osha_corpus.csv', index=False)
print('Saved.')

Index contains 4427 vectors
Saved.


## 3. Retrieval Function

A custom retrieval function is used to encode a user-given text into a vector representation using the same embedding model. The function then queries the FAISS index to find the top-k most similar documents to the user-given text. 

In [12]:
def retrieve(query, k=5):
    query = query.lower()

    query_embedding = embedder.encode([query]).astype('float32')

    distances, indices = index.search(query_embedding, k)

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        row = df.iloc[idx]
        results.append({
            'text': row['Abstract Text'],
            'injury_type': row['Event type'],
            'distance': float(dist)
        })
    
    return results

# Test
query = 'What causes hand injuries?'
for result in retrieve(query):
    print(result)

{'text': 'A 9:30 a.m. on March 15, 2017, an employee was moving a steel plate when one of the tacks broke.  The employee fractured both hands and several fingers when the steel plate fell on top of his hands. The employee was hospitalized and treated for his injuries. ', 'injury_type': 'Struck-by', 'distance': 0.8905067443847656}
{'text': "On January 11, 2017, an employee was pressing truss plates on the roller press and got his hand caught into the press track.  No definitive description of injuries were provided in initial report other than the employee's hand was crushed.  The employee was hospitalized. ", 'injury_type': 'Caught in or between', 'distance': 0.9118542075157166}
{'text': 'At 1:30 p.m. on January 23, 2017, an employee was operating a press brake bending some metal.  The employee had his hand in the press and hit the pedal, actuating the machine.  The press came down and crushed four fingers.  The employee was hospitalized and treated for his injuries. ', 'injury_type': 

## 4. Generating Answers with Claude

In order to answer user queries into our data, we generate a prompt for each query based on what is returned from our retrieval process. This information is then passed to Claude to generate a response.

In [15]:
def rag_query(query):
    # Retreive the top 5 results
    results = retrieve(query)

    context = '\n\n'.join([
        f"Injury Type: {r['injury_type']}\nText: {r['text']}"
        for r in results
    ])

    prompt = f"""
    You are an occupational health analyst. 
    Use only the workplace injury records below to answer the question. 
    If the records don't contain enough information, say you are unsure.
    
    INJURY RECORDS:
    {context}

    QUESTION: 
    {query}"""

    client = anthropic.Anthropic()  #

    response = client.messages.create(
        model='claude-sonnet-4-5',
        max_tokens=512,
        messages=[{'role': 'user', 'content': prompt}]
    )

    return response.content[0].text


In [16]:
print(rag_query('What causes hand injuries?'))

Based on the workplace injury records provided, hand injuries are caused by:

1. **Falling objects** - Heavy materials (like steel plates) falling onto hands when equipment fails (e.g., broken tacks)

2. **Machinery with moving parts** - Hands getting caught in:
   - Press tracks on roller presses
   - Press brakes during bending operations
   - Conveyor belts and tension rollers

3. **Crush hazards** - Hands being crushed between:
   - Vehicle equipment (forklifts) and fixed structures (storage racks)
   - Machine components during operation

4. **Operator actions** - Including:
   - Placing hands in the point of operation while machines are active
   - Accidental activation of controls (like pressing a pedal while hand is in the machine)
   - Attempting to manually stop equipment
   - Cleaning machinery while it's still running

The common thread in these records is hands being in dangerous positions near moving machinery or heavy materials, often during routine operations like mater

In [18]:
print(rag_query('What are common causes of fall injuries?'))

Based on the workplace injury records provided, common causes of fall injuries include:

1. **Lack of fall protection**: In at least one case, the employee was explicitly not wearing fall protection while working on a roof at elevation.

2. **Not using available safety equipment properly**: One employee was wearing a personal fall arrest system but was not tied off at the time of the fall.

3. **Loss of footing/balance**: Multiple incidents mention employees losing their footing or balance while working at elevation.

4. **Equipment failure or instability**: One incident involved a steel grate that slipped off its supporting beam.

5. **Stepping backwards without awareness**: One employee stepped backwards while on an elevated platform and fell.

6. **Misjudging grip points on ladders**: One employee reached for what he thought was the top rung of a ladder but grabbed air instead, causing loss of balance.

These falls occurred from various heights (16-27 feet) and during different acti

In [19]:
print(rag_query('What industries have the most severe injuries?'))

Based on the injury records provided, I can identify the following industries with severe injuries:

**Most Severe Injuries:**

1. **Construction/Roofing** - Multiple severe cases:
   - Roofing contractor: Employee fell from ladder, suffered 4-5 fractured ribs, two punctured lungs, fractured left ankle, and fractured orbital bone
   - Snow removal/roofing: Employee fell through roof, suffered back compression fracture (L1 vertebrae), scraped arms, and shoulder injury
   - Sign installation/construction: Employee ejected from boom lift, suffered open fractured femur, nine broken ribs, broken collar bone, punctured lung, and multiple lacerations

2. **Manufacturing** - Two severe cases:
   - Tank manufacturer: Employee struck by falling metal, suffered deep laceration and leg fractures
   - Warehouse/manufacturing: **Fatal injury** - Employee struck by falling skid of AC motors, resulting in fatal head injury

**Answer:** Based on these limited records, **construction-related industries 